# PoLaRIS / Pohang Canal - Real Data Demo

Pohang Canal Datasetの実データ（AWS S3から無料DL可能）を使った可視化デモです。

This notebook demonstrates visualization of real maritime sensor data from the Pohang Canal Dataset (free download from AWS S3).

In [ ]:
!pip install -e ".." -q

## Download Navigation Data / データダウンロード

Calibration (3KB) + Navigation (21MB) のみダウンロードします。画像・LiDARはサイズが大きいためスキップ。

In [ ]:
import subprocess
from pathlib import Path

data_dir = Path("../data/polaris_pohang00")
data_dir.mkdir(parents=True, exist_ok=True)

for f in ["calibration.zip", "navigation.zip"]:
    if not (data_dir / f).exists():
        subprocess.run([
            "aws", "s3", "cp",
            f"s3://pohang-canal-dataset/pohang00/{f}",
            str(data_dir / f),
            "--no-sign-request"
        ], check=True)
        subprocess.run(["unzip", "-o", str(data_dir / f), "-d", str(data_dir)], check=True)
        print(f"Downloaded and extracted {f}")
    else:
        print(f"{f} already exists")

## Load & Inspect GPS Data / GPSデータの読み込み

In [ ]:
from rdh.datasets.pohang_canal import load_gps, load_ahrs, load_extrinsics, latlon_to_local
import numpy as np

gps = load_gps(data_dir / "navigation" / "gps.txt")
print(f"GPS points: {len(gps)}")
print(f"Duration: {gps[-1, 0] - gps[0, 0]:.1f} seconds")
print(f"Latitude range: [{gps[:, 1].min():.6f}, {gps[:, 1].max():.6f}]")
print(f"Longitude range: [{gps[:, 2].min():.6f}, {gps[:, 2].max():.6f}]")

## GPS Trajectory / GPS軌跡

In [ ]:
import matplotlib.pyplot as plt

x, y = latlon_to_local(gps[:, 1], gps[:, 2])
t = gps[:, 0] - gps[0, 0]

fig, ax = plt.subplots(figsize=(10, 10))
speed = np.clip(np.sqrt(np.gradient(x, t)**2 + np.gradient(y, t)**2), 0, 15)
sc = ax.scatter(x, y, c=speed, s=2, cmap='RdYlGn_r')
ax.scatter(x[0], y[0], c='blue', s=100, marker='^', label='Start', zorder=5)
ax.scatter(x[-1], y[-1], c='red', s=100, marker='s', label='End', zorder=5)
ax.set_xlabel('East (m)')
ax.set_ylabel('North (m)')
ax.set_title('Pohang Canal - GPS Trajectory (speed-colored)')
ax.set_aspect('equal')
ax.legend()
ax.grid(True, alpha=0.3)
fig.colorbar(sc, label='Speed (m/s)')
plt.tight_layout()
plt.show()

## AHRS (IMU) Data / 姿勢センサデータ

In [ ]:
ahrs = load_ahrs(data_dir / "navigation" / "ahrs.txt")
t_ahrs = ahrs[:, 0] - ahrs[0, 0]
print(f"AHRS samples: {len(ahrs)}, rate: {len(ahrs) / t_ahrs[-1]:.1f} Hz")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax1.plot(t_ahrs, ahrs[:, 5], 'r-', lw=0.3, alpha=0.7, label='wx')
ax1.plot(t_ahrs, ahrs[:, 6], 'g-', lw=0.3, alpha=0.7, label='wy')
ax1.plot(t_ahrs, ahrs[:, 7], 'b-', lw=0.3, alpha=0.7, label='wz')
ax1.set_ylabel('Angular Velocity (rad/s)')
ax1.set_title('Gyroscope')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(t_ahrs, ahrs[:, 8], 'r-', lw=0.3, alpha=0.7, label='ax')
ax2.plot(t_ahrs, ahrs[:, 9], 'g-', lw=0.3, alpha=0.7, label='ay')
ax2.plot(t_ahrs, ahrs[:, 10], 'b-', lw=0.3, alpha=0.7, label='az')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Acceleration (m/s²)')
ax2.set_title('Accelerometer')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Sensor Configuration / センサ配置

In [ ]:
extr = load_extrinsics(data_dir / "calibration" / "extrinsics.json")
print(f"Sensors: {len(extr)}")
for name, data in extr.items():
    t = data['translation']
    print(f"  {name:20s}: x={t[0]:+.2f}, y={t[1]:+.2f}, z={t[2]:+.2f}")

## Full Visualization / フル可視化

`rdh demo` コマンドでも同じ可視化が可能です:  
```bash
rdh demo polaris --data-dir ../data/polaris_pohang00 --save output.png
```

In [ ]:
from rdh.datasets.pohang_canal import viz_pohang_canal
viz_pohang_canal(data_dir)